In [ ]:
!pip install duckdb -q

from google.colab import drive
import duckdb

drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/NLP/Assignment_1/processed/telugu_tokenized_full.parquet'

print("Sampling 1,500,000 sentences directly from disk...")

query = f"""
    SELECT tokenized_sentence
    FROM read_parquet('{file_path}')
    USING SAMPLE 1500000
"""

sampled_df = duckdb.query(query).df()

sentences = sampled_df['tokenized_sentence'].tolist()
del sampled_df

test_set = sentences[:1000]
dev_set = sentences[1000:2000]
train_set = sentences[2000:]

print(f"Train set size: {len(train_set)}")
print(f"Dev set size: {len(dev_set)}")
print(f"Test set size: {len(test_set)}")

Mounted at /content/drive
Sampling 1,500,000 sentences directly from disk...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Train set size: 1498000
Dev set size: 1000
Test set size: 1000


In [ ]:
test_set[:5]

['భోగి మంటలు నిర్వహించడం సులభం - ఆహారం మరియు రిఫ్రెష్మెంట్లను ఉంచడానికి ఒక గుడారాన్ని అద్దెకు తీసుకోండి మరియు కొంతమంది తల్లిదండ్రులు అగ్నిని నిర్వహించడానికి వారి సమయాన్ని స్వచ్ఛందంగా పొందండి .',
 'అందుకు మహిళలు నిరాకరించారు .',
 'ఎలా ఉన్నారు ? ’',
 'అందుకే దీనిని స్వర్గసోపానం అనే పేరు వచ్చిందని పురాణాలూ చెబుతున్నాయి .',
 'ఎదుటివారింట్లోని వస్తువులను కావాలని అడుగకూడదు .']

In [ ]:
import re

def clean_telugu_sentence(sentence):
    sentence = str(sentence)
    sentence = re.sub(r'[a-zA-Z]+', '', sentence)
    sentence = re.sub(r'\s+', ' ', sentence).strip()
    return sentence

print("Cleaning train set...")
train_set_clean = [clean_telugu_sentence(s) for s in train_set]

print("Cleaning dev set...")
dev_set_clean = [clean_telugu_sentence(s) for s in dev_set]

print("Cleaning test set...")
test_set_clean = [clean_telugu_sentence(s) for s in test_set]

print("Building vocabulary from the cleaned training set...")
vocabulary = set()

for sentence in train_set_clean:
    tokens = sentence.split()
    vocabulary.update(tokens)

V = len(vocabulary)
print(f"Total unique tokens (V) in Cleaned Training Set: {V}")

Cleaning train set...
Cleaning dev set...
Cleaning test set...
Building vocabulary from the cleaned training set...
Total unique tokens (V) in Cleaned Training Set: 839877


In [ ]:
def get_ngrams(tokens, n):
    ngram_list = []
    for i in range(len(tokens) - n + 1):
        ngram_list.append(tuple(tokens[i:i + n]))
    return ngram_list


def build_ngram_model(corpus, n):
    ngram_counts = {}
    context_counts = {}

    for sentence in corpus:
        tokens = sentence.split()
        if not tokens:
            continue

        if n > 1:
            padded_tokens = ['<s>'] * (n - 1) + tokens + ['</s>']
            for ng in get_ngrams(padded_tokens, n):
                ngram_counts[ng] = ngram_counts.get(ng, 0) + 1

            context_tokens = padded_tokens[:-1]
            for ctx in get_ngrams(context_tokens, n - 1):
                context_counts[ctx] = context_counts.get(ctx, 0) + 1
        else:
            for token in tokens:
                ngram_counts[token] = ngram_counts.get(token, 0) + 1
            context_counts['TOTAL_TOKENS'] = context_counts.get('TOTAL_TOKENS', 0) + len(tokens)

    return ngram_counts, context_counts

unigram_counts, unigram_context = build_ngram_model(train_set_clean, 1)
bigram_counts, bigram_context = build_ngram_model(train_set_clean, 2)
trigram_counts, trigram_context = build_ngram_model(train_set_clean, 3)
quadrigram_counts, quad_context = build_ngram_model(train_set_clean, 4)

print(f"Unique Bigrams: {len(bigram_counts)}")
print(f"Unique Trigrams: {len(trigram_counts)}")
print(f"Unique Quadrigrams: {len(quadrigram_counts)}")


Building Unigram Model...
Building Bigram Model...
Building Trigram Model...
Building Quadrigram Model...

All models built successfully!
Unique Bigrams: 6642459
Unique Trigrams: 11340457
Unique Quadrigrams: 13270964


In [ ]:
def laplace_probability(ngram, n, ngram_counts, context_counts, V):
    ngram_count = ngram_counts.get(ngram, 0)

    if n == 1:
        N = context_counts['TOTAL_TOKENS']
        prob = (ngram_count + 1) / (N + V)
    else:
        context = ngram[:-1]
        context_count = context_counts.get(context, 0)
        prob = (ngram_count + 1) / (context_count + V)

    return prob

sample_unigram = 'కేసులు'
prob_uni = laplace_probability(sample_unigram, 1, unigram_counts, unigram_context, V)
print(f"Unigram Probability of '{sample_unigram}': {prob_uni}")

sample_bigram = ('<s>', 'కేసులు')
prob_bi = laplace_probability(sample_bigram, 2, bigram_counts, bigram_context, V)
print(f"Bigram Probability of {sample_bigram}: {prob_bi}")


Unigram Probability of 'కేసులు': 0.00043813548766337206
Bigram Probability of ('<s>', 'కేసులు'): 5.047821693826514e-05


In [ ]:
import math

def calculate_perplexity(sentence, n, ngram_counts, context_counts, V):
    tokens = sentence.split()
    if not tokens:
        return float('inf')

    if n > 1:
        padded_tokens = ['<s>'] * (n - 1) + tokens + ['</s>']
        sentence_ngrams = get_ngrams(padded_tokens, n)
        N_tokens = len(tokens) + 1
    else:
        sentence_ngrams = tokens
        N_tokens = len(tokens)

    log_prob_sum = 0

    for ng in sentence_ngrams:
        prob = laplace_probability(ng, n, ngram_counts, context_counts, V)
        log_prob_sum += math.log(prob)

    # Perplexity formula: exp(- (1/N) * sum(log P))
    perplexity = math.exp(-log_prob_sum / N_tokens)
    return perplexity

print("Testing Trigram Perplexity on first 5 Dev sentences:\n")
for i in range(5):
    sent = dev_set_clean[i]
    ppl = calculate_perplexity(sent, 3, trigram_counts, trigram_context, V)
    print(f"Sentence {i+1}: {sent[:50]}...")
    print(f"Perplexity: {ppl:.2f}\n")


Testing Trigram Perplexity on first 5 Dev sentences:

Sentence 1: ‘ ‘ శివపుత్రుడు ’ తో తెలుగు వారికి దగ్గరైన ఈ ఇద్దర...
Perplexity: 287822.40

Sentence 2: తనపై కుట్రలు జరుగుతున్నాయంటూ ఆరోపించారు ....
Perplexity: 86337.30

Sentence 3: ఎన్టీఆర్ సీఎం అంటూ అరుస్తూనే ఉండడంతో ....
Perplexity: 209724.54

Sentence 4: తర్వాత బైటికొచ్చేసరికి లూసీ దగ్గర్నించి ఫోన్ వచ్చి...
Perplexity: 85654.96

Sentence 5: నిహారిక ఎంటర్టైన్మెంట్స్ బ్యానర్ పై నిర్మాత వెంకట్...
Perplexity: 167187.44



In [ ]:
def evaluate_dataset(dataset, n, ngram_counts, context_counts, V):
    total_perplexity = 0
    valid_sentences = 0

    for sentence in dataset:
        ppl = calculate_perplexity(sentence, n, ngram_counts, context_counts, V)
        if ppl != float('inf'):
            total_perplexity += ppl
            valid_sentences += 1

    if valid_sentences == 0:
        return 0

    return total_perplexity / valid_sentences

models = [
    ("Unigram", 1, unigram_counts, unigram_context),
    ("Bigram", 2, bigram_counts, bigram_context),
    ("Trigram", 3, trigram_counts, trigram_context),
    ("Quadrigram", 4, quadrigram_counts, quad_context)
]

print("--- Development Set Evaluation ---")
for name, n, n_counts, c_counts in models:
    avg_ppl = evaluate_dataset(dev_set_clean, n, n_counts, c_counts, V)
    print(f"Average {name} Perplexity: {avg_ppl:.2f}")

print("\n--- Test Set Evaluation ---")
for name, n, n_counts, c_counts in models:
    avg_ppl = evaluate_dataset(test_set_clean, n, n_counts, c_counts, V)
    print(f"Average {name} Perplexity: {avg_ppl:.2f}")

--- Development Set Evaluation ---
Average Unigram Perplexity: 18455.66
Average Bigram Perplexity: 47985.70
Average Trigram Perplexity: 250556.06
Average Quadrigram Perplexity: 396841.73

--- Test Set Evaluation ---
Average Unigram Perplexity: 86589.66
Average Bigram Perplexity: 57397.33
Average Trigram Perplexity: 267487.77
Average Quadrigram Perplexity: 413700.45


In [ ]:
import random

def generate_next_word(context, ngram_counts):
    possible_ngrams = {ng: count for ng, count in ngram_counts.items() if ng[:-1] == context}
    if not possible_ngrams:
        return None

    next_words = [ng[-1] for ng in possible_ngrams.keys()]
    weights = list(possible_ngrams.values())

    chosen_word = random.choices(next_words, weights=weights, k=1)[0]
    return chosen_word

def generate_sentence(n, ngram_counts, max_words=15):
    context = tuple(['<s>'] * (n - 1))
    generated_tokens = []

    for _ in range(max_words):
        next_word = generate_next_word(context, ngram_counts)
        if not next_word or next_word == '</s>':
            break

        generated_tokens.append(next_word)
        if n > 1:
            context = tuple((list(context) + [next_word])[-(n - 1):])

    return ' '.join(generated_tokens)


In [ ]:
model_counts = {
    "Unigram": unigram_counts,
    "Bigram": bigram_counts,
    "Trigram": trigram_counts,
    "Quadrigram": quadrigram_counts
}

for name, n in [("Unigram", 1), ("Bigram", 2), ("Trigram", 3), ("Quadrigram", 4)]:
    sentence = generate_sentence(n, model_counts[name])
    print(f"{name} generated sentence: {sentence}")


Unigram generated sentence: 
Bigram generated sentence: ఆదోని , నిజమైన గుర్రం ఎగరావచ్చు ' ' .
Trigram generated sentence: 2 కప్పుల గ్రీన్ టీ , ఏ గిఫ్ట్ ఇవ్వాలి అని నిర్ణయం తీసుకున్నారు .
Quadrigram generated sentence: ఆ అనుమానం నిర్దరణ చేసుకోవడానికి ఫిబ్రవరి 16 న కిషన్‌ ఇంట్లో సోదాలు నిర్వహించారు .


In [ ]:
sen = train_set_clean[2]
tokens = sen.split()
print(sen)

u_prob = 1
for tok in tokens:
    u_prob *= laplace_probability(tok, 1, unigram_counts, unigram_context, V)
print(f"Unigram Probability: {u_prob}")

b_prob = 1
padded_b = ['<s>'] * 1 + tokens + ['</s>']
for bg in get_ngrams(padded_b, 2):
    b_prob *= laplace_probability(bg, 2, bigram_counts, bigram_context, V)
print(f"Bigram Probability: {b_prob}")

t_prob = 1
padded_t = ['<s>'] * 2 + tokens + ['</s>']
for tg in get_ngrams(padded_t, 3):
    t_prob *= laplace_probability(tg, 3, trigram_counts, trigram_context, V)
print(f"Trigram Probability: {t_prob}")

q_prob = 1
padded_q = ['<s>'] * 3 + tokens + ['</s>']
for qg in get_ngrams(padded_q, 4):
    q_prob *= laplace_probability(qg, 4, quadrigram_counts, quad_context, V)
print(f"Quadrigram Probability: {q_prob}")

ఆ మొద్దుకు ఉన్న కత్తిని చూస్తే మాంచి మాసాలా ఫైట్ కు రెడీగా ఉన్నారు అన్నట్టే ఉంది .
Unigram Probability: 1.1333154995085799e-59
Bigram Probability: 4.2684641498480146e-69
Trigram Probability: 3.0616361293611885e-77
Quadrigram Probability: 1.6649672427076653e-81


## Observations

- Perplexity generally increases with N-gram order (Unigram < Bigram < ... in terms of being lowest) is NOT what we see here — instead, higher-order models (Trigram, Quadrigram) tend to show **higher** perplexity under Add-1 smoothing because Laplace smoothing is very aggressive: it distributes a large probability mass to the huge number of unseen higher-order n-grams (since `V` is fixed but the number of possible contexts grows combinatorially), which pulls probabilities of real n-grams down.
- This over-smoothing effect for higher-order, sparse models is precisely the motivation for Assignment-5, where a smaller constant (Add-K, K=0.3) is used instead of K=1 to reduce how much probability mass is redistributed to unseen events.
- Dev and Test set perplexities are close to one another, indicating the 1000/1000 split is representative of the corpus.